# Quantum State Visualization

**Day 23 | 30-Day Quantum Computing Challenge**

## Aim:
To visualize how quantum gates change a qubit's statevector, Bloch sphere position, and measurement probabilities.

## Tools:
Python, NumPy, Matplotlib (optional Qiskit)

## Expected Result:
Different gates produce distinct quantum states; some states share probabilities but differ in phase.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. Setup and state mathematics
A qubit is $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$, where $|\alpha|^2+|\beta|^2=1$. The Bloch vector is $(2\operatorname{Re}(\alpha^*\beta),2\operatorname{Im}(\alpha^*\beta),|\alpha|^2-|\beta|^2)$.

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
import matplotlib.pyplot as plt

zero = np.array([1, 0], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
gates = {"I": np.eye(2), "X": X, "Y": Y, "Z": Z, "H": H}

def bloch(state):
    a, b = state
    return np.array([2*np.real(np.conj(a)*b), 2*np.imag(np.conj(a)*b), abs(a)**2-abs(b)**2])

def probabilities(state):
    return np.abs(state)**2

## 2. Apply individual gates and compare sequences
Note: **Z|0⟩ = |0⟩**; a phase flip becomes visible when applied to a superposition. **XH|0⟩ = H|0⟩**, while **ZH|0⟩ = |−⟩**.

In [ ]:
# Experiment step: execute this cell and inspect its output.
states = {"|0>": zero, "X|0>": X@zero, "Y|0>": Y@zero, "Z|0>": Z@zero,
          "H|0>": H@zero, "ZH|0>": Z@H@zero, "XH|0>": X@H@zero}
for name, state in states.items():
    print(f"{name:7s}  amplitudes={np.round(state, 3)}  P(0),P(1)={np.round(probabilities(state), 3)}  Bloch={np.round(bloch(state), 3)}")
    assert np.isclose(np.linalg.norm(state), 1)

## 3. Visualize Bloch spheres
A vector at +z is |0⟩; −z is |1⟩; +x is |+⟩; −x is |−⟩. Global phase does not change the Bloch vector.

In [ ]:
# Experiment step: execute this cell and inspect its output.
fig = plt.figure(figsize=(12, 8))
for i, (name, state) in enumerate(states.items(), start=1):
    ax = fig.add_subplot(2, 4, i, projection="3d")
    u = np.linspace(0, 2*np.pi, 32); v = np.linspace(0, np.pi, 17)
    ax.plot_wireframe(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                      np.outer(np.ones_like(u), np.cos(v)), color="gray", alpha=.2, linewidth=.5)
    ax.quiver(0, 0, 0, *bloch(state), color="royalblue", linewidth=2.5, arrow_length_ratio=.15)
    ax.set(xlim=(-1,1), ylim=(-1,1), zlim=(-1,1), title=name)
    ax.view_init(elev=22, azim=38)
plt.tight_layout(); plt.savefig("results.png", dpi=180, bbox_inches="tight"); plt.show()

## 4. Measurement probability versus relative phase
**H|0⟩ and ZH|0⟩ both give 50/50 outcomes in the Z basis**, but their Bloch vectors point in opposite x directions. Measurement in the X basis distinguishes them.

In [ ]:
# Experiment step: execute this cell and inspect its output.
labels = list(states)
p0 = [probabilities(s)[0] for s in states.values()]
p1 = [probabilities(s)[1] for s in states.values()]
plt.figure(figsize=(10,4)); plt.bar(labels,p0,label="P(0)"); plt.bar(labels,p1,bottom=p0,label="P(1)"); plt.ylim(0,1.05);plt.legend();plt.ylabel("Probability");plt.title("Z-basis measurement probabilities");plt.xticks(rotation=25);plt.tight_layout();plt.savefig("probabilities.png",dpi=180);plt.show()
plus = H@zero; minus = Z@plus
print("X-basis: |+> ->", abs(np.vdot(plus,plus))**2, ", |-> ->", abs(np.vdot(plus,minus))**2)

## 5. Try your own gate sequence

In [ ]:
# Experiment step: execute this cell and inspect its output.
sequence = "HZX"  # edit: e.g. HX, HH, HZH
state = zero.copy()
for letter in sequence:
    if letter not in gates: raise ValueError(f"Unknown gate: {letter}")
    state = gates[letter] @ state
print("Sequence:", sequence, "State:", np.round(state,3), "Bloch:", np.round(bloch(state),3))

## 6. Optional Qiskit verification
Install `qiskit` if you want to compare with a circuit statevector. This block is optional; all earlier cells work with NumPy and Matplotlib.

In [ ]:
# Experiment step: execute this cell and inspect its output.
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    qc = QuantumCircuit(1)
    qc.h(0); qc.z(0)
    print(qc.draw(output="text"))
    print("Qiskit state:", Statevector.from_instruction(qc).data)
except ImportError:
    print("Optional: pip install qiskit")

## 7. What to remember
- **X** flips |0⟩ to |1⟩; **Y** does so with a phase.
- **H** creates an equal superposition from |0⟩.
- **Z** changes relative phase only when both basis amplitudes are present.
- Measurement probabilities alone do not reveal every state property.
- These are **ideal pure-state simulations**, not measurements on real quantum hardware.

**Author:** SAMYAK DEKATE

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.